In [ ]:
!pip install transformers torchaudio librosa pandas tqdm soundfile

['Requirement already satisfied: transformers in /usr/local/lib/python3.12/dist-packages (5.0.0)',
 'Requirement already satisfied: torchaudio in /usr/local/lib/python3.12/dist-packages (2.10.0+cpu)',
 'Requirement already satisfied: librosa in /usr/local/lib/python3.12/dist-packages (0.11.0)',
 'Requirement already satisfied: pandas in /usr/local/lib/python3.12/dist-packages (2.2.2)',
 'Requirement already satisfied: tqdm in /usr/local/lib/python3.12/dist-packages (4.67.3)',
 'Requirement already satisfied: soundfile in /usr/local/lib/python3.12/dist-packages (0.13.1)',
 'Requirement already satisfied: filelock in /usr/local/lib/python3.12/dist-packages (from transformers) (3.29.0)',
 'Requirement already satisfied: huggingface-hub<2.0,>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (1.11.0)',
 'Requirement already satisfied: numpy>=1.17 in /usr/local/lib/python3.12/dist-packages (from transformers) (2.0.2)',
 'Requirement already satisfied: packaging>=20.0 in /

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import librosa
import pandas as pd
from tqdm import tqdm
from transformers import HubertForCTC, Wav2Vec2Processor

# =====================================================
# CONFIG
# =====================================================

DATASET_DIR = "/content/drive/MyDrive/IIITH_Text/TESS Toronto emotional speech set data"

OUTPUT_CSV = "tess_full_metadata.csv"
FAILED_LOG = "failed_files.txt"

MODEL_NAME = "facebook/hubert-large-ls960-ft"

# =====================================================
# LOAD HUBERT MODEL
# =====================================================

print("Loading HuBERT model...")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = HubertForCTC.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print(f"Using device: {device}")

# =====================================================
# TRANSCRIBE FUNCTION
# =====================================================

def transcribe_audio(audio_path):

    # Load audio as 16kHz mono
    audio, sr = librosa.load(audio_path, sr=16000)

    # Prepare input
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(device)

    # Inference
    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    transcription = processor.batch_decode(predicted_ids)[0]

    return transcription.lower()

# =====================================================
# PROCESS DATASET
# =====================================================

rows = []
failed_files = []

folders = sorted(os.listdir(DATASET_DIR))

sample_id = 0

for folder in folders:

    folder_path = os.path.join(DATASET_DIR, folder)

    if not os.path.isdir(folder_path):
        continue

    print(f"\nProcessing folder: {folder}")

    files = sorted(os.listdir(folder_path))

    for file in tqdm(files):

        if not file.endswith(".wav"):
            continue

        filepath = os.path.join(folder_path, file)

        try:

            # =================================================
            # Example filename:
            # OAF_bath_fear.wav
            # =================================================

            filename = file.replace(".wav", "")

            parts = filename.split("_")

            speaker = parts[0]
            word = parts[1]
            emotion = parts[2]

            # =================================================
            # TRANSCRIBE AUDIO
            # =================================================

            transcription = transcribe_audio(filepath)

            # =================================================
            # SAVE ROW
            # =================================================

            rows.append({
                "id": sample_id,
                "filepath": filepath,
                "filename": file,
                "speaker": speaker,
                "word": word,
                "emotion": emotion,
                "transcription": transcription
            })

            sample_id += 1

        except Exception as e:

            failed_files.append(filepath)

            print(f"\nError processing: {filepath}")
            print(e)

# =====================================================
# CREATE DATAFRAME
# =====================================================

df = pd.DataFrame(rows)

# =====================================================
# SAVE CSV
# =====================================================

df.to_csv(OUTPUT_CSV, index=False)

print("\n=================================================")
print("DONE!")
print("Saved metadata CSV:", OUTPUT_CSV)
print("Total processed files:", len(df))
print("Failed files:", len(failed_files))
print("=================================================")

# =====================================================
# SAVE FAILED FILES
# =====================================================

with open(FAILED_LOG, "w") as f:

    for item in failed_files:
        f.write(item + "\n")

print(f"Failed file log saved: {FAILED_LOG}")

# =====================================================
# SHOW SAMPLE OUTPUT
# =====================================================

print("\nSample rows:")
print(df.head())

Loading HuBERT model...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Using device: cpu

Processing folder: OAF_Fear


100%|██████████| 200/200 [07:24<00:00,  2.22s/it]



Processing folder: OAF_Pleasant_surprise


...


Processing folder: YAF_pleasant_surprised


100%|██████████| 200/200 [07:38<00:00,  2.29s/it]



Processing folder: YAF_sad


100%|██████████| 200/200 [08:04<00:00,  2.42s/it]


DONE!
Saved metadata CSV: tess_full_metadata.csv
Total processed files: 2800
Failed files: 0
Failed file log saved: failed_files.txt

Sample rows:
   id                                           filepath           filename  \
0   0  /content/drive/MyDrive/IIITH_Text/TESS Toronto...  OAF_back_fear.wav   
1   1  /content/drive/MyDrive/IIITH_Text/TESS Toronto...   OAF_bar_fear.wav   
2   2  /content/drive/MyDrive/IIITH_Text/TESS Toronto...  OAF_base_fear.wav   
3   3  /content/drive/MyDrive/IIITH_Text/TESS Toronto...  OAF_bath_fear.wav   
4   4  /content/drive/MyDrive/IIITH_Text/TESS Toronto...  OAF_bean_fear.wav   

  speaker  word emotion      transcription  
0     OAF  back    fear  say the word back  
1     OAF   bar    fear   say the word bar  
2     OAF  base    fear  say the word base  
3     OAF  bath    fear  say the word bath  
4     OAF  bean    fear  say the word bein  


In [ ]:
import os
import pickle
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "/content/drive/MyDrive/IIITH_Text/tess_full_metadata.csv"

SAVE_DIR = "/content/drive/MyDrive/IIITH_Text"

MAX_TOKENS = 8
BATCH_SIZE = 32
EPOCHS = 30

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

print("\nDataset Shape:", df.shape)

# ============================================================
# USE TRANSCRIPTION COLUMN
# ============================================================

# Example:
# "say the word bath"

texts = df["transcription"].astype(str)

labels = df["emotion"]

# ============================================================
# ENCODE LABELS
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(labels)

df["y"] = y

print("\nEmotion Classes:")
print(label_encoder.classes_)

# Save label encoder
with open(f"{SAVE_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["y"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["y"],
    random_state=42
)

print("\n====================================")
print("Train Size:", len(train_df))
print("Validation Size:", len(val_df))
print("Test Size:", len(test_df))
print("====================================")

# ============================================================
# SAVE SPLITS
# ============================================================

train_df.to_csv(f"{SAVE_DIR}/text_train_split.csv", index=False)
val_df.to_csv(f"{SAVE_DIR}/text_val_split.csv", index=False)
test_df.to_csv(f"{SAVE_DIR}/text_test_split.csv", index=False)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(train_df["transcription"].astype(str))

# Save tokenizer
with open(f"{SAVE_DIR}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# ============================================================
# TEXT -> SEQUENCES
# ============================================================

X_train = tokenizer.texts_to_sequences(
    train_df["transcription"].astype(str)
)

X_val = tokenizer.texts_to_sequences(
    val_df["transcription"].astype(str)
)

X_test = tokenizer.texts_to_sequences(
    test_df["transcription"].astype(str)
)

# ============================================================
# PAD SEQUENCES
# ============================================================

X_train = pad_sequences(
    X_train,
    maxlen=MAX_TOKENS,
    padding="post"
)

X_val = pad_sequences(
    X_val,
    maxlen=MAX_TOKENS,
    padding="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_TOKENS,
    padding="post"
)

vocab_size = len(tokenizer.word_index) + 1

print("\nVocabulary Size:", vocab_size)

# ============================================================
# BUILD MODEL
# ============================================================

def build_model(vocab_size, max_tokens, num_classes):

    inp = layers.Input(
        shape=(max_tokens,),
        name="text_input"
    )

    # ========================================================
    # Embedding Layer
    # ========================================================

    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True,
        name="text_embedding"
    )(inp)

    # ========================================================
    # BiLSTM
    # ========================================================

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            dropout=0.3,
            recurrent_dropout=0.3
        ),
        name="contextual_modelling"
    )(x)

    # ========================================================
    # Dense Layer
    # ========================================================

    x = layers.Dense(
        128,
        activation="relu",
        name="text_representation"
    )(x)

    x = layers.Dropout(0.4)(x)

    # ========================================================
    # Output Layer
    # ========================================================

    out = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = models.Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]
    )

    return model

# ============================================================
# CREATE MODEL
# ============================================================

model = build_model(
    vocab_size=vocab_size,
    max_tokens=MAX_TOKENS,
    num_classes=len(label_encoder.classes_)
)

# ============================================================
# MODEL SUMMARY
# ============================================================

print("\nModel Summary:\n")

model.summary()

# ============================================================
# CALLBACKS
# ============================================================

cb = [

    callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True,
        monitor="val_accuracy"
    ),

    callbacks.ModelCheckpoint(
        filepath=f"{SAVE_DIR}/best_text_model.keras",

        save_best_only=True,

        monitor="val_accuracy"
    )
]

# ============================================================
# TRAIN MODEL
# ============================================================

history = model.fit(

    X_train,
    train_df["y"].values,

    validation_data=(
        X_val,
        val_df["y"].values
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=cb
)

# ============================================================
# EVALUATE
# ============================================================

loss, accuracy = model.evaluate(
    X_test,
    test_df["y"].values
)

print("\n====================================")
print("TEST ACCURACY:", accuracy)
print("====================================")

# ============================================================
# PREDICTIONS
# ============================================================

pred_probs = model.predict(X_test)

preds = pred_probs.argmax(axis=1)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:\n")

print(classification_report(
    test_df["y"].values,
    preds,
    target_names=label_encoder.classes_
))

# ============================================================
# SAVE FINAL MODEL
# ============================================================

model.save(f"{SAVE_DIR}/final_text_model.keras")

print("\nModel Saved!")


Dataset Shape: (2800, 7)

Emotion Classes:
['angry' 'disgust' 'fear' 'happy' 'neutral' 'ps' 'sad']

Train Size: 2023
Validation Size: 357
Test Size: 420

Vocabulary Size: 363

Model Summary:



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_embedding      │ (None, 8, 64)     │     23,232 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 8)         │          0 │ text_input[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ contextual_modelli… │ (None, 128)       │     66,048 │ text_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_representation │ (None, 128)       │     16,512 │ contextual_model… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ text_representat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 7)         │        903 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 106,695 (416.78 KB)

 Trainable params: 106,695 (416.78 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 14s 114ms/step - accuracy: 0.1216 - loss: 1.9476 - val_accuracy: 0.1317 - val_loss: 1.9462
Epoch 2/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - accuracy: 0.1270 - loss: 1.9467 - val_accuracy: 0.1429 - val_loss: 1.9467
Epoch 3/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.1409 - loss: 1.9463 - val_accuracy: 0.1401 - val_loss: 1.9473
Epoch 4/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 103ms/step - accuracy: 0.1597 - loss: 1.9452 - val_accuracy: 0.0756 - val_loss: 1.9494
Epoch 5/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 0.1666 - loss: 1.9433 - val_accuracy: 0.1232 - val_loss: 1.9536
Epoch 6/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 85ms/step - accuracy: 0.1750 - loss: 1.9375 - val_accuracy: 0.0840 - val_loss: 1.9728
Epoch 7/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.2042 - loss: 1.9205 - val_accuracy: 0.0476 - val_loss: 2.0297
Epoch 8/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - accuracy: 0.2190 - loss: 1.8915 - val_accuracy: 0.0588

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
